In [ ]:
# RadarNetCDFloader.ipynb

# OPENING IMPORTS

import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

import zipfile as zp          # used for unzipping ppi files
from pathlib import Path      # used to play with pathnames to save 
from datetime import datetime # used to manipulate time :)

import wradlib as wr          # used for having fun with radar data

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

import h5py                   # used for reading .h5 files (Radar Level 1 data)
import h5netcdf               # used for converting .h5 files to NetCDF

# TAKEN FROM "Part4IntroductionToGridding"
import cartopy.crs as ccrs
import pyart
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker



In [ ]:
# SPECIAL METHOD TO IMPORT LEROI RADAR GRIDDING PACKAGE FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks')
from leroi.leroi import *

In [ ]:
# FUNCTION
#pcolormesh but takes 1D X and Y coordinates for centres of the pixels

def pcolormeshC(x_centers, y_centers, z, ax=None,
                            shading='auto', **pcolor_kwargs):
    """
    Create a pcolormesh from a 2D array and 1D coordinate-center arrays.

    Parameters
    ----------
    x_centers : 1D array
        X coordinates of cell centers (length = number of columns in z)
    y_centers : 1D array
        Y coordinates of cell centers (length = number of rows in z)
    z : 2D array
        Data array with shape (len(y_centers), len(x_centers))
    ax : matplotlib.axes.Axes, optional
        Existing axis to draw on
    shading : str
        Passed to pcolormesh (default: 'auto')
    **pcolor_kwargs
        Extra kwargs passed to pcolormesh

    Returns
    -------
    pcm : QuadMesh
        The pcolormesh object
    """

    x_centers = np.asarray(x_centers)
    y_centers = np.asarray(y_centers)
    z = np.asarray(z)

    if z.shape != (len(y_centers), len(x_centers)):
        raise ValueError(
            f"z shape {z.shape} does not match "
            f"(len(y_centers), len(x_centers)) = "
            f"({len(y_centers)}, {len(x_centers)})"
        )

    # Convert centers -> edges
    def centers_to_edges(c):
        dc = np.diff(c)

        edges = np.empty(len(c) + 1)

        # Interior edges
        edges[1:-1] = c[:-1] + dc / 2

        # Extrapolate outer edges
        edges[0] = c[0] - dc[0] / 2
        edges[-1] = c[-1] + dc[-1] / 2

        return edges

    x_edges = centers_to_edges(x_centers)
    y_edges = centers_to_edges(y_centers)

    if ax is None:
        fig, ax = plt.subplots()

    pcm = ax.pcolormesh(
        x_edges,
        y_edges,
        z,
        shading=shading,
        **pcolor_kwargs
    )

    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    return pcm

In [ ]:
# VERTICAL CROSS SECTION PLOTTING
# THIS BLOCK LOADS IN FROM NET CDF FILES STORED IN SCRATCH

# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# PLEASE MAKE IT SO THE PLOT VARIABLES CAN BE CHOSEN HERE!!!! (ADD STRING EXECUTERS AND SUCH)
# PlotType = 'Vert'
# PlotVar  = 'corrected_reflectivity'
# Slice    = str(EWsliceKM) + 'kmEW'

# the reference number for the radar location (ie 22 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'ppi'

if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9

# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)

RadarFileDate  = YYYY + MM + DD 

houri = 12
mini = 50


# THIS BLOCK IS ABOUT THE VERTICAL CROSS SECTION SLICE NORTH OR SOUTH OF THE RADAR
# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES

EWsliceKM = 20 # [km]     # number of km north or south of the radar you want to take the east-west slice for x-section
EWsliceNorS = 'North'    # direction ['North' or 'South'] from the radar you want the slice taken

# add a positive or negative sign to the slice for math
if (EWsliceNorS == 'South'):
    EWsliceKMsign = EWsliceKM * -1
elif (EWsliceNorS == 'North'):
    EWsliceKMsign = EWsliceKM * 1
else:
    print("Please choose EWsliceNorS to be 'North' or 'South'.")


# for houri in range(0,24):
#     for mini in range(0,60,5):
        
RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 

# add a string of format hh:mm:ss for printing
print('working on ' + RadarFileTimePrint)




xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
                        RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')

# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES

yvalsKM = np.array(xgrid.x) * 0.001             # x coordinates in km

# farthest south and north y-values
MinSliceKM = int(np.min(yvalsKM))
MaxSliceKM = int(np.max(yvalsKM))

# for EWsliceKM in range(MinSliceKM, MaxSliceKM+1):
#     print('working on slice at ' + str(EWsliceKMsign) + ' km' )

# find the vertical cross section index in the coordinates
EWslicei = np.where(yvalsKM == EWsliceKMsign)[0][0]  # index in the x-coordinates where that north-south km value lives

# Create a vertical plot of the reflectivity for a 5-min period
    
fig, ax = plt.subplots(figsize=(8,6))
GridViewer = pcolormeshC(xgrid.x*0.001, xgrid.z*0.001, xgrid.corrected_specific_differential_phase[0,:,EWslicei,:], ax=ax, cmap='nipy_spectral', vmin=0, vmax=90)
# mutiply by 0.001 to get distances in km                                        # THIS SLICE COMES FROM 50 KM NORTH OF THE RADAR TO BETTER SEE THE VOLUME

ax.set_xlabel('East-West Distance [km]')
ax.set_ylabel('Altitude Above Radar [km]')
plt.title('Reflectivity Cross Section for ' + RadarSiteName + ' Radar\n For Slice Taken ' + \
      str(EWsliceKM) + ' km ' + EWsliceNorS + ' of the Radar on\n' + \
      RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
      str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')
plt.colorbar(GridViewer, ax=ax, label = 'Reflectivity [dBZ]')
plt.grid()

plt.xlim([MinSliceKM, MaxSliceKM])
plt.ylim([0,20])

# SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
# SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_' + PlotType + Slice + '.png'

# SavePath = SaveFolder + SaveFile

# if not Path(SaveFolder).exists():
#     print('Creating Folder: ' + SaveFolder)
#     Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
# plt.close()

In [ ]:
# HORIZONTAL CROSS SECTION PLOTTING
# THIS BLOCK LOADS IN FROM NET CDF FILES STORED IN SCRATCH

# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# PLEASE MAKE IT SO THE PLOT VARIABLES CAN BE CHOSEN HERE!!!! (ADD STRING EXECUTERS AND SUCH)
# PlotType = 'Vert'
# PlotVar  = 'corrected_reflectivity'
# Slice    = str(EWsliceKM) + 'kmEW'

# the reference number for the radar location (ie 22 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'ppi'

if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14

# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)

RadarFileDate  = YYYY + MM + DD 

houri = 12
mini = 00

# for houri in range(0,24):
#     for mini in range(0,60,5):
        
RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 

# add a string of format hh:mm:ss for printing
print('working on ' + RadarFileTimePrint)

xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
                        RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')



#CHANGE THE TIME INDICES
#CHANGE THE TIME INDICES
#CHANGE THE TIME INDICES
#CHANGE THE TIME INDICES
#CHANGE THE TIME INDICES
#CHANGE THE TIME INDICES


# ADD LABELS FOR ALTITUDE AND SUCH
# ADD LABELS FOR ALTITUDE AND SUCH
# ADD LABELS FOR ALTITUDE AND SUCH
# ADD LABELS FOR ALTITUDE AND SUCH
# ADD LABELS FOR ALTITUDE AND SUCH
# ADD LABELS FOR ALTITUDE AND SUCH
# ADD LABELS FOR ALTITUDE AND SUCH
# ADD LABELS FOR ALTITUDE AND SUCH

fig, ax = plt.subplots(figsize=(8,6))
GridViewer = pcolormeshC(xgrid.x*0.001, xgrid.y*0.001, xgrid.corrected_reflectivity[0,5,:,:], ax=ax, cmap='nipy_spectral', vmin=-10, vmax=50)
# mutiply by 0.001 to get distances in km

ax.set_xlabel('East-West Distance [km]')
ax.set_ylabel('North-South Distance [km]')
plt.title('Radar Site ' + str(RadarIDno) + ' Reflectivity on ' + RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
          str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6])
plt.colorbar(GridViewer, ax=ax, label = 'Reflectivity [dBZ]')
plt.grid()

# SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/'
# SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + GridOrPPI + '500m.png'

# SavePath = SaveFolder + SaveFile

# if not Path(SaveFolder).exists():
#     print('doing')
#     Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)

In [ ]:
xgrid